
# Geodatabase Timeseries with the OpenET API

This notebook provides a step-by-step guide on utilizing the OpenET API's */geodatabase/metadata/ids* endpoint to retrieve field IDs contained within a specified region. Then make an additional query to the */geodatabase/timeseries* endpoint to get monthly evapotranspiration and precipitation data for all fields within

## To complete this notebook you will need:
1. [OpenET Account](https://auth.etdata.org/)
2. [OpenET API Key](https://auth.etdata.org/dashboard)
3. A valid GeoJSON file or Shapefile. This demo uses the HUC-12 watershed boundary of the Lower Little Shasta River.

## Additional Resources

[OpenET API Reference](https://openet.gitbook.io/docs/reference/api-reference)

___

# Purpose

* **Evaluate the impact of irrigation curtailments on consumptive use within the Lower Little Shasta River watershed**
* [Example Use Case/Study in northern CA for Shasta Valley](https://doi.org/10.1016/j.jhydrol.2025.134119)
    * Surface diversions and groundwater curtailed in order of water rights priority<br><br>
    * [2021/2022 timeline](https://ucanr.edu/sites/default/files/2024-01/393149.pdf)<br><br>
    * 2022 Curtailments - California's State Water Resources Control Board (SWRCB) [curtailment orders](https://www.waterboards.ca.gov/drought/scott_shasta_rivers/shasta_addendums.html) were in effect multiple times to protect instream flows<br><br>
    * Late August 2022 - large diversion operated in violation of curtailment orders<br><br>
    * 2020 - the last typical pre-curtailment irrigation season that the SWRCB defined as a baseline against which Scott Valley farmers' 2022 groundwater conservation agreements were evaluated

___

# Collab Notebook Outline

1.   **Import Python packages**

2. **Configure API Key**: Copy your API Key and paste in prompt to access OpenET data.

3.   **Load boundary as GeoJSON**: Load a GeoJson and create temporary asset ID to pass Geometry to API.

4.  **Make /geodatabase/metadata/ids API request** to retrieve the field ids associated with all fields within the watershed.

5.  **Make /geodatabase/timeseries API request** to save time series of evapotranspiration and precipitation for all fields within the watershed: Construct and send a request to retrieve ET, and Pr data for all field boundaries within the defined region of interest.

6. **Visualize impact of curtailments on field-level irrigation consumptive use.**

7. **Retrieve streamflow data to check if flows increased during curtailment year**
___

# 1. Import Python packages

In [ ]:
# !pip install geopandas pandas matplotlib dataretrieval requests shapely folium gdown

from pathlib import Path
import glob
import getpass
import gzip
import io
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import requests
import shapely
import folium
from dataretrieval import waterdata

# ---------------------------------
# Downloads Siskiyou Land Use/Water Source Type geojson
# ---------------------------------
!gdown 11adW0EbIaY2WeNE9tOmaLW-hWK7g-0nl

# ---------------------------------
# Spatial aggregation boundaries
# ---------------------------------
#  Downloads Lower Little Shasta River HUC-12 Boundary geojson
!gdown 1JE3YcfCGSMr59I1uVm5BORhdpk700hTu
# Downloads Montague Water District Boundary geojson
# !gdown 1fyQxPL_2nViBLCFGFddq_uqUyo_YNj3Y

# 2. Configure your API Key to access OpenET data

#### Go to your [OpenET API dashboard](https://auth.etdata.org/dashboard) page and copy the API Key.

#### Then paste your API Key below after running this cell

In [2]:

YOUR_API_KEY = getpass.getpass("Paste your API key here --> ")

header = {"Authorization": YOUR_API_KEY}

Paste your API key here -->  ········


# 3. Load boundary from GeoJSON or Shapefile

In [3]:
# -----------------------------
# specify the filename to look for
# -----------------------------

# -----------------------------
# HUC-12 Lower Little Shasta River Watershed
# -----------------------------
filename = "workshop_llsr.geojson"

# -----------------------------
# Montague Water District Boundary
# -----------------------------
# filename = "i03_WaterDistricts_Montague.geojson"


## Visualize the watershed boundary along with land use/irrigaiton water system type data

In [ ]:

# current working directory
current_path = Path.cwd()

# file path from the specified file name
file_path = current_path / 'geospatial_data' / filename

file_prefix = filename.split('.')[0]
file_type = filename.split('.')[-1]

if file_type == 'geojson' or file_type == 'zip':

    try:
        bound_gdf = gpd.read_file(file_path)

        is_projected = bound_gdf.crs.is_projected
        if is_projected:
            print('reprojecting the boundary CRS to EPSG:4326 WGS84 GCS')
            bound_gdf.crs = "EPSG:3857" 
            bound_gdf = bound_gdf.to_crs(4326)

        # export a shapefile version of the geojson if it doesn't exist already
        shp_path = current_path / 'geospatial_data' / f"{file_prefix}.shp"
        
        if not shp_path.exists():
            print('saving a shapefile version of the provided boundary')
            bound_gdf.to_file(shp_path, driver="ESRI Shapefile")
            
    except Exception as e:
        print(e)

elif file_type == 'shp':

    try:
        bound_gdf = gpd.read_file(file_path)

        is_projected = bound_gdf.crs.is_projected
        if is_projected:
            print('reprojecting the boundary CRS to EPSG:4326 WGS84 GCS')
            bound_gdf.crs = "EPSG:3857" 
            bound_gdf = bound_gdf.to_crs(4326)
    
    except Exception as e:
        print(e)   

else:
    print('cannot recognize the file type provided, please ensure your file type is either a geojson, shapefile, or zipped shapefile')

# -------------------
# create the map
# -------------------
f = folium.Figure(width=800, height=500)
m = folium.Map(location=[41.72, -122.4419], zoom_start=12).add_to(f)

# -------------------
# add basemap
# -------------------
tile = folium.TileLayer(
        tiles = 'https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}',
        attr = 'Google',
        name = 'Google Satellite Hybrid',
        overlay = False,
        control = True
       ).add_to(m)

# -------------------
# add watershed boundary
# -------------------
bound_map_lyr = folium.GeoJson(
    bound_gdf,
    name="Watershed Boundary",
    style_function=lambda feature: {
        "fillColor": "red",
        "color": "black",
        "weight": 2,
        "fillOpacity": 0.5,
    },
)

bound_map_lyr.add_to(m)

# -------------------
# Land Use/Water Source Type dataset
# -------------------
wr_data_path =  current_path / 'geospatial_data' / "i15_LandUse_Siskiyou2010.geojson"
wr_data = gpd.read_file(wr_data_path).to_crs(4326)

# -------------------
# keep polygons that intersect the watershed boundary only
# -------------------
intersected_gdf = gpd.sjoin(wr_data, bound_gdf, how="inner", predicate="intersects")
wr_data = wr_data[wr_data.index.isin(intersected_gdf.index)]

# format the source type data
wr_dat_dict = {"1": 1, "3": 3}
wr_data['SOURCE'] = wr_data['WATERSOURC'].map(wr_dat_dict)
wr_data['SOURCE'] = wr_data['SOURCE'].fillna(9)
wr_data = wr_data.reset_index()

wr_data["SOURCE_label"] = (
    wr_data["SOURCE"]
    .astype(str)
    .str.strip()
    .map({
        "1": "Surface water",
        "3": "Groundwater",
        # "9": "Other",
    })
    .fillna("Unknown")
)

# Categorical colors
source_colors = {
    "Surface water": "#1f78b4",  # blue
    "Groundwater": "#33a02c",    # green
    # "Other": "#ff7f00",          # orange
}


def source_style(feature):
    source = feature["properties"].get("SOURCE_label", "Unknown")

    return {
        "fillColor": source_colors.get(source, "#bdbdbd"),
        "color": "#333333",
        "weight": 0.7,
        "fillOpacity": 0.65,
    }

def highlight_style(feature):
    return {
        "fillColor": "#ffff00",
        "color": "#000000",
        "weight": 2,
        "fillOpacity": 0.85,
    }

# ----------------------
# add the irrigation water source types to the map
# ----------------------
wr = folium.GeoJson(
    wr_data,
    name="Irrigation Water Source Type",
    style_function=source_style,
    highlight_function=highlight_style,
    tooltip=folium.GeoJsonTooltip(
        fields=["SOURCE_label"],
        aliases=["Water source:"],
        localize=True,
        sticky=False,
        labels=True,
    ),
    popup=folium.GeoJsonPopup(
        fields=["SOURCE_label"],
        aliases=["Water source:"],
        localize=True,
        labels=True,
    ),
)
    
wr.add_to(m)

# ------------------------------------------------------------
# Add a categorical legend
# ------------------------------------------------------------
legend_html = """
<div style="
    position: fixed;
    bottom: 5px;
    left: 5px;
    z-index: 9999;
    background-color: white;
    border: 2px solid #777;
    border-radius: 5px;
    padding: 10px;
    font-size: 14px;
">
    <b>Irrigation Water Source Type</b><br>
    <i style="background:#1f78b4; width:14px; height:14px; display:inline-block;"></i>
    Surface water<br>
    <i style="background:#33a02c; width:14px; height:14px; display:inline-block;"></i>
    Groundwater<br>
</div>
"""

m.get_root().html.add_child(folium.Element(legend_html))

# Add layer control and display map in Jupyter
folium.LayerControl().add_to(m)

# Extract the coordinates of the GeoJSON with shapely.get_coordinates to a list to pass to the API.
# The code flattens the matrix and puts the coordinates into a 1D list (format required for the API).
boundary_geom = shapely.get_coordinates(bound_gdf.geometry).flatten().tolist()

# initialize the map
m

# 4. Retrieve OpenET Field IDs contained within the Boundary

In [ ]:

# -------------------
# This provides the API with your API key.
# -------------------
api_header = {"AUTHORIZATION": YOUR_API_KEY}

# -------------------
# This provides the parameters for the request.
# -------------------
api_query = {
    "geometry": boundary_geom
}

# california server
api_request = requests.post(
    url="https://openet-api-california-470570065811.us-west1.run.app/geodatabase/metadata/ids",
    headers=api_header,
    json=api_query
)
# main server
# api_request = requests.post(
#     url="https://openet-api.org/geodatabase/metadata/ids",
#     headers=api_header,
#     json=api_query
# )

# -------------------
# Parse the compressed response
# -------------------
fields = eval(gzip.decompress(api_request.content).decode())

print(f"Length of filtered list: {len(fields)}")
print(f"List of filtered field IDs: {fields}")

In [37]:
# -----------------------
# limit to 100 features for the API request if needed
# -----------------------
# fields = fields[0:99]

# -----------------------
# test two fields instead of 180 if needed
# -----------------------
# fields = [21106018259, 21106016902]


## Retrieve field boundary metadata properties/attributes
**crop types, area, etc.**

In [ ]:

# endpoint arguments
args_prop = {
  "field_ids":fields
}

# california server
resp_prop = requests.post(
    headers=api_header,
    json=args_prop,
    url="https://openet-api-california-470570065811.us-west1.run.app/geodatabase/metadata/properties"
)
# resp_prop = requests.post(
#     headers=api_header,
#     json=args_prop,
#     url="https://openet-api.org/geodatabase/metadata/properties"
# )

# parse the compressed response
metadata_properties = eval(gzip.decompress(resp_prop.content).decode())

print(f"Field properties: {metadata_properties[0]}")

## Retrieve field boundary geometries and visualize crop types

In [ ]:

# endpoint arguments
args_geom = {
  "field_ids":fields
}

# query the api
# california server
resp_geom = requests.post(
    headers=api_header,
    json=args_geom,
    url="https://openet-api-california-470570065811.us-west1.run.app/geodatabase/metadata/boundaries"
)
# main server
# resp_geom = requests.post(
#     headers=api_header,
#     json=args_geom,
#     url="https://openet-api.org/geodatabase/metadata/boundaries"
# )

# unzip the data
geom_data = eval(gzip.decompress(resp_geom.content).decode())

# -------------------
# create the map
# -------------------
f = folium.Figure(width=800, height=500)
m = folium.Map(location=[41.72, -122.4419], zoom_start=12).add_to(f)

# -------------------
# add basemap
# -------------------
tile = folium.TileLayer(
        tiles = 'https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}',
        attr = 'Google',
        name = 'Google Satellite Hybrid',
        overlay = False,
        control = True
       ).add_to(m)

# -------------------
# add watershed boundary
# -------------------
bound_map_lyr = folium.GeoJson(
    bound_gdf,
    name="Watershed Boundary",
    style_function=lambda feature: {
        # "fillColor": "red",
        "color": "black",
        "weight": 2,
        "fillOpacity": 0.5,
    }
)
bound_map_lyr.add_to(m)

# -------------------
# convert metadata to a dataframe
# -------------------
meta_gdf = pd.DataFrame.from_records(metadata_properties)[["field_id", "crop_2022"]]

# -------------------
# convert geometries to a geodataframe
# -------------------
fb_geom_gdf = gpd.GeoDataFrame.from_features(geom_data["features"])
fb_geom_gdf.crs = 4326

# -------------------
# merge crop types with geometries
# -------------------
fb_geom_gdf = fb_geom_gdf.merge(meta_gdf, on='field_id', how='left')

# convert Field ID to string for tooltip
fb_geom_gdf['field_id'] = fb_geom_gdf['field_id'].astype(str)

# -------------------
# create new labels for crop types
# -------------------
fb_geom_gdf["crop_2022_l"] = (
    fb_geom_gdf["crop_2022"]
    .astype(str)
    .str.strip()
    .map({
        "25": "25 - Other Small Grains",
        "36": "36 - Alfalfa",
        "37": "37 - Other Hay/Non Alfalfa",
        "61": "61 - Fallow/Idle Cropland",
        "176": "176 - Grassland/Pasture",
    })
)

# -------------------
# colors to map for crop types
# -------------------
crop_colors = {
    "25 - Other Small Grains": "yellow",
    "176 - Grassland/Pasture": "purple",
    "36 - Alfalfa": "pink",
    "37 - Other Hay/Non Alfalfa": "palegreen",
    "61 - Fallow/Idle Cropland": "brown", 
}


def crop_style(feature):
    crop = feature["properties"].get("crop_2022_l")

    return {
        "fillColor": crop_colors.get(crop, "#bdbdbd"),
        "color": "#333333",
        "weight": 0.7,
        "fillOpacity": 0.65,
    }

def highlight_style(feature):
    return {
        "fillColor": "blue",
        "color": "#000000",
        "weight": 2,
        "fillOpacity": 0.85,
    }

# ----------------------
# add the crop types to the map
# ----------------------
fields_map_lyr = folium.GeoJson(
    fb_geom_gdf,
    name="Crop Type",
    style_function=crop_style,
    highlight_function=highlight_style,
    tooltip=folium.GeoJsonTooltip(
        fields=["field_id", "crop_2022_l"],
        aliases=["Field ID:", "2022 Crop Type:"],
        localize=True,
        sticky=False,
        labels=True,
    ),
    popup=folium.GeoJsonPopup(
        fields=["field_id", "crop_2022_l"],
        aliases=["Field ID:", "2022 Crop Type:"],
        localize=True,
        labels=True,
    ),
)
    
fields_map_lyr.add_to(m)


folium.LayerControl().add_to(m)

# initialize the map
m


## 5. Retrieve monthly time series data from the API for 2016-2025
**Available Data Variables**:
> ET - OpenET actual ET<br>
> PR - gridMET precipitation<br>
> ETO - Bias corrected gridMET grass reference ET<br>
> NDVI - Normalized Difference Vegetation Index<br>
> ETOF - Fraction of reference ET (ET / ETO)<br>

In [15]:

# -----------------------------
# specify dates to retrieve data for
# -----------------------------
start_date = "2016-01-01"
end_date = "2025-12-01"

# -----------------------------
# specify an OpenET model (ENSEMBLE, DISALEXI, EEMETRIC, GEESEBAL, PTJPL, SIMS, SSEBOP)
# -----------------------------
model = "ENSEMBLE"

# -----------------------------
# list of data variables
# -----------------------------
variables = ["ET", "ETO", "PR", "NDVI", "ETOF"]
# variables = ["ET", "PR"]

# -----------------------------
# time step of the data (monthly or daily)
# -----------------------------
time_step = "monthly"

# -----------------------------
# output file format (JSON or CSV) - CSV fails for large # of fields
# -----------------------------
output_file_format = "JSON"

# query the api
api_ts_query = {
    "field_ids": fields,
    "date_range": [start_date, end_date],
    "interval": "monthly",
    "variables": variables,
    "models": [model.lower()],
    "file_format": output_file_format
}

# california server
api_ts_request = requests.post(
    url="https://openet-api-california-470570065811.us-west1.run.app/geodatabase/timeseries",
    headers=api_header,
    json=api_ts_query
)
# main server
# api_ts_request = requests.post(
#     url="https://openet-api.org/geodatabase/timeseries",
#     headers=api_header,
#     json=api_query
# )

# -----------------------------
# 'value_mm' units are in mm
# -----------------------------
if output_file_format == "JSON":
    timeseries = eval(gzip.decompress(api_ts_request.content).decode())
    display(timeseries[0])
elif output_file_format == "CSV":
    timeseries = api_ts_request.content.decode()
    display(timeseries)

{'time': '2016-01-01',
 'field_id': 21106016347,
 'collection': 'ensemble_et',
 'value_mm': 13.39}

## Read the time series data as a Pandas dataframe

In [ ]:

# -----------------------------
#read dataframe from json or csv
# -----------------------------
if output_file_format == "JSON":
    df_meta = pd.DataFrame.from_records(metadata_properties)[["field_id", "hectares"]]

    df = pd.DataFrame.from_records(timeseries)

elif output_file_format == "CSV":
    df_meta = pd.DataFrame.from_records(metadata_properties)[["field_id", "hectares"]]

    df = pd.read_csv(io.StringIO(timeseries))

# -------------------
# calculate field acreage from hectares
# -------------------
df_meta["acres"] = (
    df_meta["hectares"] * 2.471
)

# convert time string column to datetime object
df["time"] = pd.to_datetime(df["time"])

# -------------------
# pivot the time series
# -------------------
df_wide = df.pivot(index=["time", "field_id"], columns="collection", values="value_mm")
df_wide = df_wide.reset_index()

# rename data columns
naming_dict = {
    f"{model.lower()}_et": "et",
    "gridmet_pr": "pr",
    "gridmet_eto": "eto",
    "landsat_ndvi": "ndvi",
    f"{model.lower()}_etof": "etof"
}
df_wide = df_wide.rename(columns=naming_dict)

# -------------------
# merge the field acreages and time series
# -------------------
df_all = pd.merge(df_wide, df_meta[["field_id", "acres"]], how="inner", on=["field_id"])

# add the year as a column
df_all['year'] = df_all.time.dt.year

display(df_all.head())


# Calculate monthly Net ET (ET - P).clip(lower=0):

While there are more advanced approaches. This method assumes:
* 100% of precipitation is during growing season
* minimal to 0 soil water carry over
* no shallow groundwater contributions that offset irrigation consumptive use

## Aggregate growing season ET, Net ET, and P volumes for individual fields and for the watershed

In [ ]:

df_all = df_all.copy()

# -------------------
# date info
# -------------------
df_all["time"] = pd.to_datetime(df_all["time"])
df_all["year"] = df_all["time"].dt.year
df_all["month"] = df_all["time"].dt.month

# -------------------
# filter dataframe to Mar-Oct
# -------------------
df_m_to_o = df_all.loc[
    df_all["month"].between(3, 10)
].copy()

# -------------------
# calculate volumes in acft from depth in mm
# -------------------
df_m_to_o["et_acft"] = (
    df_m_to_o["et"] / 304.8
) * df_m_to_o["acres"]

df_m_to_o["pr_acft"] = (
    df_m_to_o["pr"] / 304.8
) * df_m_to_o["acres"]

# -------------------
# ET - P = Net ET
# -------------------
df_m_to_o["net_et_acft"] = (
    df_m_to_o["et_acft"] - df_m_to_o["pr_acft"]
).clip(lower=0)

# inspect the table
display(df_m_to_o.query("net_et_acft > 0").head())

# -------------------
# export the field level monthly timeseries to CSV
# -------------------
df_m_to_o.to_csv(current_path / 'tables' / 'openet_field_summaries_monthly_2016_2025.csv', index=False)


# ----------------------------------------
# Growing season aggregation of field-level monthly totals
# ----------------------------------------
etn_yearly_sums = (
    df_m_to_o
        .groupby(["field_id", "year"])[["field_id", "year", "acres", "net_et_acft", "et_acft", "pr_acft"]]
        .agg({
            "acres": "first", 
            "net_et_acft": "sum", 
            "et_acft": "sum", 
            "pr_acft": "sum",
        })
        .reset_index()
)

# ----------------------------------------
# Spatial aggregation of field-level growing season totals
# ----------------------------------------
etn_yearly_sums_agg = (
    etn_yearly_sums
        .rename(columns={"field_id": "field_count", "acres": "total_acres"})
        .groupby("year", as_index=False)
        .agg({
            "total_acres": "sum",
            "net_et_acft": "sum",
            "et_acft": "sum",
            "pr_acft": "sum",
            "field_count": "nunique",
        })
        .sort_values("year")
)

# -------------------
# calculate area-weighted average rates
# -------------------
etn_yearly_sums_agg["net_et_mm"] = (
    etn_yearly_sums_agg["net_et_acft"]
    / etn_yearly_sums_agg["total_acres"]
    * 304.8
)

etn_yearly_sums_agg["et_mm"] = (
    etn_yearly_sums_agg["et_acft"]
    / etn_yearly_sums_agg["total_acres"]
    * 304.8
)

etn_yearly_sums_agg["pr_mm"] = (
    etn_yearly_sums_agg["pr_acft"]
    / etn_yearly_sums_agg["total_acres"]
    * 304.8
)

etn_yearly_sums_agg["net_et_inches"] = (
    etn_yearly_sums_agg["net_et_acft"]
    / etn_yearly_sums_agg["total_acres"]
    * 12
)

etn_yearly_sums_agg["et_inches"] = (
    etn_yearly_sums_agg["et_acft"]
    / etn_yearly_sums_agg["total_acres"]
    * 12
)

etn_yearly_sums_agg["pr_inches"] = (
    etn_yearly_sums_agg["pr_acft"]
    / etn_yearly_sums_agg["total_acres"]
    * 12
)

etn_yearly_sums_agg = etn_yearly_sums_agg.set_index('year')

# -------------------
# export aggregated annual time series to CSV
# -------------------
etn_yearly_sums_agg.to_csv(current_path / 'tables' / 'openet_watershed_summaries_annual_2016_2025.csv')

display(etn_yearly_sums_agg.head())

## Plot Time Series of Growing Season ET, Net ET, and PPT volumes for the watershed boundary

In [ ]:

plt.style.use("ggplot")

fig, ax_et = plt.subplots(figsize=(9, 5))
ax_precip = ax_et.twinx()

# Primary y-axis: Net ET and ET
etn_yearly_sums_agg[["et_acft", "net_et_acft"]].plot(
    ax=ax_et,
    color=["green", "chocolate"],
    linewidth=2,
)

# Secondary y-axis: precipitation
etn_yearly_sums_agg[["pr_acft"]].plot(
    ax=ax_precip,
    color="steelblue",
    linewidth=2,
    legend=False
)

# -----------------------------
# Formatting
# -----------------------------
ax_et.set_ylabel("Mar–Oct ET / Net ET volume (acre-feet)")
ax_precip.set_ylabel("Mar–Oct precipitation volume (acre-feet)")
ax_precip.tick_params(axis="y")

ax_et.set_xlabel("")
ax_et.set_title("Lower Little Shasta River Irrigation Consumptive Use Volumes")

# for the training example
if len(fields) == 180:
    ax_et.set_ylim(0, 30000)
    ax_precip.set_ylim(0, 12000)
else:
    max_et = etn_yearly_sums_agg['et_acft'].max() + 5000
    ax_et.set_ylim(0, max_et)
    max_pr = etn_yearly_sums_agg['pr_acft'].max() + 2000
    ax_precip.set_ylim(0, max_pr)

# Combine legend entries from both axes
lines_1, labels_1 = ax_et.get_legend_handles_labels()
lines_2, labels_2 = ax_precip.get_legend_handles_labels()

ax_et.legend(
    lines_1 + lines_2,
    labels_1 + labels_2,
    loc="lower left",
)

fig.tight_layout()
plt.show()

## Plot timeseries of monthly area-weighted average rates

In [ ]:

# sum volumes grouped by each date
etn_monthly_sums = (
    df_m_to_o
        .groupby(["time"])[["acres", "net_et_acft", "et_acft", "pr_acft"]]
        .sum()
        .reset_index()
)

# calculate average rates in inches
etn_monthly_sums['net_et_inches'] = (
    etn_monthly_sums['net_et_acft']
    / etn_monthly_sums['acres'] 
    * 12
)

etn_monthly_sums['et_inches'] = (
    etn_monthly_sums['et_acft']
    / etn_monthly_sums['acres'] 
    * 12
)

etn_monthly_sums['pr_inches'] = (
    etn_monthly_sums['pr_acft']
    / etn_monthly_sums['acres'] 
    * 12
)

plt.style.use("ggplot")
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["grid.color"] = "#E5E5E5"
plt.rcParams["axes.edgecolor"] = "#CBCBCB"  # Light grey border color
plt.rcParams["axes.linewidth"] = 1.0        # Thickness of the border

# ------------------------------------------------------------
# plotting columns
# ------------------------------------------------------------
df = etn_monthly_sums.copy()

date_column = "time"

et_column = "et_inches"
net_et_column = "net_et_inches"
precip_column = "pr_inches"

analysis_months = list(range(3, 11))  # March–October
month_labels = ["Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct"]


# ------------------------------------------------------------
# prepare dates and add year/month fields
# ------------------------------------------------------------
if date_column is not None:
    df[date_column] = pd.to_datetime(df[date_column])
    df = df.set_index(date_column)
else:
    df.index = pd.to_datetime(df.index)

df["year"] = df.index.year
df["month_number"] = df.index.month

df = df.loc[df["month_number"].isin(analysis_months)].copy()


# ------------------------------------------------------------
# create an Month × Year table for a variable
# ------------------------------------------------------------
def make_month_year_table(data, value_column):
    return (
        data.pivot_table(
            index="month_number",
            columns="year",
            values=value_column,
            aggfunc="sum",
        )
        .reindex(analysis_months)
    )


et_by_year = make_month_year_table(df, et_column)
net_et_by_year = make_month_year_table(df, net_et_column)
precip_by_year = make_month_year_table(df, precip_column)


# ------------------------------------------------------------
# Plot: ET | Net ET | Precipitation
# ------------------------------------------------------------
fig, axes = plt.subplots(
    nrows=1,
    ncols=3,
    figsize=(16, 5),
    sharey=True,
    constrained_layout=True,
)

plot_specs = [
    (axes[0], et_by_year, "ET", "tab:red"),
    (axes[1], net_et_by_year, "Net ET (ET − P)", "tab:purple"),
    (axes[2], precip_by_year, "Precipitation", "tab:blue"),
]

for ax, plot_data, title, color in plot_specs:
    for year in plot_data.columns:
        if year == 2022:
            color = "red"
            linewidth = 2.8
            alpha = 1.0
            zorder = 3
            label = "2022 (curtailment)"
        elif year == 2020:
            color = "black"
            linewidth = 2.4
            alpha = 1.0
            zorder = 2
            label = "2020 (baseline)"
        elif year == 2025:
            color = "lightgray"
            linewidth = 1.0
            alpha = 0.8
            zorder = 1
            label = "Other years:\n2016, 2017, 2018, 2019,\n2021, 2022, 2023, 2024,\n2025"
        else:
            color = "lightgray"
            linewidth = 1.0
            alpha = 0.8
            zorder = 1
            label = "_nolegend_"  # Do not add every background year to legend
    
        ax.plot(
            plot_data.index,
            plot_data[year],
            color=color,
            marker="o",
            markersize=4,
            linewidth=linewidth,
            alpha=alpha,
            zorder=zorder,
            label=label,
        )

    ax.set_title(title)
    ax.set_xlabel("Month")
    ax.set_xticks(analysis_months)
    ax.set_xticklabels(month_labels)
    ax.set_xlim(2, 11)
    ax.grid(alpha=0.3)

axes[0].set_ylabel("Depth (inches/month)")

# One shared legend outside the third panel
handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    title="Year",
    bbox_to_anchor=(1.01, 0.95),
    loc="upper left",
)

fig.suptitle(
    "Lower Little Shasta River Monthly Irrigation Consumptive Use by Year",
    fontsize=14,
)

plt.show()

## Calculate absolute difference and % change in ET and Net ET volumes from the baseline year (i.e., 2020)

In [ ]:
# -----------------------------
# get annual data totals for select analysis years (2020 and 2022)
# -----------------------------
et_2020 = etn_yearly_sums_agg.loc[etn_yearly_sums_agg.index == 2020]['et_acft'].item()
net_2020 = etn_yearly_sums_agg.loc[etn_yearly_sums_agg.index == 2020]['net_et_acft'].item()

et_2022 = etn_yearly_sums_agg.loc[etn_yearly_sums_agg.index == 2022]['et_acft'].item()
net_2022 = etn_yearly_sums_agg.loc[etn_yearly_sums_agg.index == 2022]['net_et_acft'].item()

# ---------------------
# curtailment year (2022) minus baseline year (2020)
# ---------------------
et_abs_diff = et_2022 - et_2020
net_abs_diff = net_2022 - net_2020

# ---------------------
# percent change
# ---------------------
et_per_diff = ((et_2022 - et_2020) / et_2020) * 100
net_per_diff = ((net_2022 - net_2020) / net_2020) * 100

print(f"ET Volume 2020 Total: {et_2020:,.2f} acre-feet")
print(f"ET Volume 2022 Total: {et_2022:,.2f} acre-feet")
print(f"ET Volume Absolute Difference: {et_abs_diff:,.2f} acre-feet")
print(f"ET Volume Percent Change: {et_per_diff:,.2f}%")

print(f"\nNet ET Volume 2020 Total: {net_2020:,.2f} acre-feet")
print(f"Net ET Volume 2022 Total: {net_2022:,.2f} acre-feet")
print(f"Net ET Volume Absolute Difference: {net_abs_diff:,.2f} acre-feet")
print(f"Net ET Volume Percent Change: {net_per_diff:,.2f}%")

## 6. Visualize curtailment impact on field-level irrigation consumptive use

In [ ]:

# ---------------------
# pivot the time series data
# ---------------------
etn_yearly_sums['year'] = etn_yearly_sums['year'].astype(str)

df_pivot = (
    etn_yearly_sums
        .pivot(index="field_id", columns="year", values=["et_acft", "net_et_acft", "pr_acft"])
        .reset_index()
)
df_pivot.columns = ['_'.join(col).strip() for col in df_pivot.columns.values]
df_pivot = df_pivot.rename(columns={'field_id_': 'field_id'})

# ---------------------
# create geodataframe from feature collection
# ---------------------
fields_gdf = gpd.GeoDataFrame.from_features(geom_data)
fields_gdf.crs = 4326
# join stats back to geometries
joined = fields_gdf.merge(df_pivot, on='field_id', how='inner')

# ---------------------
# comparison stats for 2020 and 2022 
# ---------------------
joined['et_abs_diff'] = (
    joined['et_acft_2022'] 
    - joined['et_acft_2020']
)
joined['et_net_abs_diff'] = (
    joined['net_et_acft_2022'] 
    - joined['net_et_acft_2020']
)

joined['ET Reduction (%)'] = (
    round(
        ((joined['et_acft_2022'] - joined['et_acft_2020']) 
        / joined['et_acft_2020']) 
        * 100 * -1, 2)
)
joined['Net ET Reduction (%)'] = (
    round(
        ((joined['net_et_acft_2022'] - joined['net_et_acft_2020'])
         / joined['net_et_acft_2020']) 
        * 100 * -1, 2)
)

joined['net_et_acft_2020'] = round(joined['net_et_acft_2020'], 2)
joined['net_et_acft_2022'] = round(joined['net_et_acft_2022'], 2)

# ---------------------
# create the map
# ---------------------
f = folium.Figure(width=800, height=500)
m = folium.Map(location=[41.72, -122.4419], zoom_start=12).add_to(f)

# ---------------------
# style the colorbar
# ---------------------
css_style = """
<style>
    .legend i {
        font-size: 16px !important; /* Enlarge tick text */
    }
    .legend {
        font-size: 18px !important; /* Enlarge title text */
        line-height: 24px !important;
    }
    /* Shift the main caption text upwards */
    g.caption text {
        font-size: 15px !important;
        dy: -10px !important; 
    }
    /* Shift and style tick marks text */
    g.legend-tick text {
        font-size: 14px !important;
        dy: 20px !important;
    }
</style>
"""
m.get_root().header.add_child(folium.Element(css_style))

# ---------------------
# add satellite basemap
# ---------------------
sat_tile = folium.TileLayer(
        tiles = 'https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}',
        attr = 'Google',
        name = 'Google Satellite Hybrid',
        overlay = False,
        control = True
       )
sat_tile.add_to(m)

# ---------------------
# add streetmap
# ---------------------
str_tile = folium.TileLayer(
    "openstreetmap", 
    name="OpenStreetMap", 
    control=True
)
str_tile.add_to(m)

def highlight_style(feature):
    return {
        "fillColor": "#ffff00",
        "color": "#000000",
        "weight": 2,
        "fillOpacity": 0.85,
    }

custom_thresholds = [0, 5, 20, 40, 60, 80, 100]

# ---------------------
# choropleth map of growing season Net ET reduction % from 2020 to 2022
# ---------------------
net_c = folium.Choropleth(
    geo_data=joined,
    name="Net ET",
    data=joined,
    columns=["field_id", "Net ET Reduction (%)"],
    key_on="feature.properties.field_id",
    fill_color="inferno_r",
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Net ET 2022 Reduction from 2020 Baseline (%)",
    highlight=True,
    threshold_scale=custom_thresholds,
)
net_c.color_scale.width = 450
net_c.add_to(m)

# ---------------------
# add nearby stream gauge marker
# ---------------------
gauge = folium.Marker(
    [41.70899, -122.53805],
    name="Stream Gauge",
    pop="Shasta River near Montague, CA",
    icon=folium.Icon(
        color="darkblue",
        icon_color="white",
        icon="info-circle",
        prefix="fa",
    ),
    tooltip="Shasta River Stream Gauge near Montague, CA"
)
gauge.add_to(m)

# hover tooltips
folium.GeoJsonTooltip(['field_id', 'Net ET Reduction (%)',  "net_et_acft_2020", "net_et_acft_2022"]).add_to(net_c.geojson)

# control box for basemap/layers
folium.LayerControl().add_to(m)

m

## 7. Retrieve streamflow data to check if flows increased during curtailment year
* [USGS-11517000 Shasta R NR Montague CA Gauge](https://waterdata.usgs.gov/monitoring-location/USGS-11517000#dataTypeId=continuous-00065-0&period=P7D&showFieldMeasurements=true)

In [ ]:

# ------------------------------------------------------------
# Site, dates, and parameter
# ------------------------------------------------------------
site_no = "11517000"  # Shasta River near Montague, CA, downstream of Little Lower Shasta River tributary
# site_no = "11517500"  # Shasta River near Yreka, CA

monitoring_location_id = f"USGS-{site_no}"

start_date = "2016-01-01"
end_date = "2025-12-31"

# USGS parameter code:
# 00060 = streamflow/discharge, generally returned in cfs
parameter_code = "00060"

# Unit conversion
CFS_TO_CUBIC_METERS_PER_SECOND = 0.028316846592


# ------------------------------------------------------------
# Retrieve daily streamflow
# ------------------------------------------------------------
daily_flow, daily_metadata = waterdata.get_daily(
    monitoring_location_id=monitoring_location_id,
    parameter_code=parameter_code,
    time=f"{start_date}/{end_date}",
)

print("Returned columns:", daily_flow.columns.tolist())
display(daily_flow.head())

if daily_flow.empty:
    raise ValueError("No daily streamflow data were returned.")

if not {"time", "value"}.issubset(daily_flow.columns):
    raise ValueError(
        "Expected 'time' and 'value' columns were not found. "
        f"Available columns: {daily_flow.columns.tolist()}"
    )


# ------------------------------------------------------------
# Clean and prepare daily streamflow data
# ------------------------------------------------------------
streamflow_daily = (
    daily_flow
    .assign(time=lambda df: pd.to_datetime(df["time"]))
    .set_index("time")[["value"]]
    .rename(columns={"value": "streamflow_cfs"})
    .sort_index()
)

streamflow_daily["streamflow_cfs"] = pd.to_numeric(
    streamflow_daily["streamflow_cfs"],
    errors="coerce",
)

# Convert cfs to cubic meters per second.
streamflow_daily["streamflow_m3s"] = (
    streamflow_daily["streamflow_cfs"]
    * CFS_TO_CUBIC_METERS_PER_SECOND
)

# display(streamflow_daily.head())


# ------------------------------------------------------------
# Create monthly mean streamflow data (m³/s)
# ------------------------------------------------------------
# The daily observations are daily mean discharge values. Averaging them
# within a month produces the monthly mean discharge in m³/s.
streamflow_monthly = (
    streamflow_daily["streamflow_m3s"]
    .resample("MS")  # MS = month-start timestamp
    .mean()
    .to_frame(name="streamflow_m3s")
)

# ------------------------------------------------------------
# Create monthly streamflow volume (acre-feet/month)
# ------------------------------------------------------------
# Daily streamflow values are daily mean cfs.
# One cfs sustained for one day equals 1.983471 acre-feet.
CFS_DAY_TO_ACRE_FEET = 86_400 / 43_560

streamflow_daily["streamflow_acft_day"] = (
    streamflow_daily["streamflow_cfs"] * CFS_DAY_TO_ACRE_FEET
)

# Sum daily volumes within each calendar month.
streamflow_monthly_volume = (
    streamflow_daily["streamflow_acft_day"]
    .resample("MS")
    .sum(min_count=1)
    .to_frame(name="streamflow_acft")
)

# Track missing daily observations by month.
monthly_day_counts = (
    streamflow_daily["streamflow_m3s"]
    .resample("MS")
    .count()
    .to_frame(name="days_with_data")
)

monthly_summary = streamflow_monthly.join(monthly_day_counts)

monthly_summary["days_in_month"] = monthly_summary.index.days_in_month
monthly_summary["complete_month"] = (
    monthly_summary["days_with_data"]
    == monthly_summary["days_in_month"]
)

# display(monthly_summary.head())


# ------------------------------------------------------------
# Analysis and curtailment settings
# ------------------------------------------------------------
comparison_years = [2020, 2022]

# Limit seasonal analysis and plots to March through October.
analysis_months = list(range(3, 11))

# ------------------------------------------------------------
# Prepare monthly March–October VOLUME plot data
# ------------------------------------------------------------
monthly_plot = streamflow_monthly_volume.copy()

monthly_plot["year"] = monthly_plot.index.year
monthly_plot["month"] = monthly_plot.index.month

# Assign all monthly data to one dummy year so the lower plot's
# month positions align visually with the upper daily plot.
monthly_plot["plot_date"] = pd.to_datetime(
    "2001-" + monthly_plot.index.strftime("%m-%d")
)

# Include all years, restricted to March–October.
monthly_by_year = (
    monthly_plot.loc[
        monthly_plot["month"].isin(analysis_months)
    ]
    .pivot(
        index="plot_date",
        columns="year",
        values="streamflow_acft",
    )
    .sort_index()
)


month_labels = [
    "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct",
]


# ------------------------------------------------------------
# Prepare 2016–2025 daily median hydrograph
# ------------------------------------------------------------
daily_plot = streamflow_daily.copy()

# Remove leap day so all years align on the same March–October axis.
daily_plot = daily_plot.loc[
    ~(
        (daily_plot.index.month == 2)
        & (daily_plot.index.day == 29)
    )
].copy()

daily_plot["year"] = daily_plot.index.year
daily_plot["month_day"] = daily_plot.index.strftime("%m-%d")

# Daily median for each calendar day across all available 2016–2025 data.
# daily_median = (
#     daily_plot
#     .groupby("month_day")["streamflow_m3s"]
#     .median()
#     .rename("median_daily_flow_m3s")
#     .reset_index()
# )
daily_median = (
    daily_plot
    .groupby("month_day")["streamflow_cfs"]
    .median()
    .rename("median_daily_flow_cfs")
    .reset_index()
)

# Use a shared non-leap year solely for plotting Jan–Dec dates together.
daily_median["plot_date"] = pd.to_datetime(
    "2001-" + daily_median["month_day"]
)

daily_median = (
    daily_median
    .sort_values("plot_date")
    .set_index("plot_date")
)

# 7-day centered moving average OF the 2016–2025 daily median.
# daily_median["median_7day_ma_m3s"] = (
#     daily_median["median_daily_flow_m3s"]
#     .rolling(window=7, center=True, min_periods=1)
#     .mean()
# )
daily_median["median_7day_ma_cfs"] = (
    daily_median["median_daily_flow_cfs"]
    .rolling(window=7, center=True, min_periods=1)
    .mean()
)


# ------------------------------------------------------------
# Prepare raw daily flow curves for 2020 and 2022
# ------------------------------------------------------------
# Prepare daily curves for every available year.
daily_year_curves = {}

plot_years = sorted(streamflow_daily.index.year.unique())

for year in plot_years:
    year_data = daily_plot.loc[daily_plot["year"] == year].copy()

    if year_data.empty:
        print(f"Warning: No daily data found for {year}.")
        continue

    # No moving average: retain the original daily mean discharge values.
    # Convert dates to the shared plotting year for overlaying 2020 and 2022.
    year_data["plot_date"] = pd.to_datetime(
        "2001-" + year_data.index.strftime("%m-%d")
    )

    daily_year_curves[year] = year_data

# ------------------------------------------------------------
# Create cumulative March–October streamflow volume curves
# ------------------------------------------------------------
plot_start = pd.Timestamp("2001-03-01")
plot_end = pd.Timestamp("2001-10-31")

cumulative_year_curves = {}

for year in plot_years:
    # Start from the original daily time series for this year.
    year_data = streamflow_daily.loc[
        streamflow_daily.index.year == year
    ].copy()

    # Keep March 1 through October 31 only.
    seasonal_data = year_data.loc[
        (year_data.index.month >= 3)
        & (year_data.index.month <= 10)
    ].copy()

    # Confirm the series actually contains data before proceeding.
    if seasonal_data.empty:
        continue

    # Cumulative volume beginning on March 1 of each year.
    seasonal_data["cumulative_streamflow_acft"] = (
        seasonal_data["streamflow_acft_day"].cumsum()
    )

    # Convert to a common non-leap dummy year for overlay plotting.
    seasonal_data["plot_date"] = pd.to_datetime(
        "2001-" + seasonal_data.index.strftime("%m-%d")
    )

    cumulative_year_curves[year] = seasonal_data

# ------------------------------------------------------------
# Plot monthly and daily comparisons
# ------------------------------------------------------------
colors = {
    2020: "gray",
    2022: "tab:blue",
}

# ------------------------------------------------------------
# Plot daily and monthly streamflow comparisons
# ------------------------------------------------------------
fig, (ax_daily, ax_monthly, ax_cumulative) = plt.subplots(
    nrows=3,
    ncols=1,
    figsize=(12, 12),
    sharex=False,  # Do not share the x-axis
    constrained_layout=True,
)

# ---- Panel A: Daily streamflow ----

# Faint gray daily lines for all years except 2020 and 2022.
for year in plot_years:
    if year in comparison_years or year not in daily_year_curves:
        continue

    year_data = daily_year_curves[year]

    if year == plot_years[0]:
        ax_daily.plot(
            year_data["plot_date"],
            year_data["streamflow_cfs"],
            color="lightgray",
            linewidth=0.8,
            alpha=0.55,
            zorder=1,
            label="Other years",
        )
    else:
        ax_daily.plot(
            year_data["plot_date"],
            year_data["streamflow_cfs"],
            color="lightgray",
            linewidth=0.8,
            alpha=0.55,
            zorder=1,
            label="_nolegend_",
        )

# 7-day moving average of the 2016–2025 daily median.
# ax_daily.plot(
#     daily_median.index,
#     daily_median["median_7day_ma_m3s"],
#     color="grey",
#     # linestyle="--",
#     linewidth=2.3,
#     zorder=3,
#     label="2016–2025 daily median, 7-day moving average",
# )
ax_daily.plot(
    daily_median.index,
    daily_median["median_7day_ma_cfs"],
    color="grey",
    # linestyle="--",
    linewidth=2.3,
    zorder=3,
    label="2016–2025 daily median, 7-day moving average",
)

# Highlight the baseline and curtailment years.
daily_styles = {
    2020: {
        "color": "steelblue",
        "linewidth": 1.5,
        "alpha": 0.95,
        "label": "2020 daily mean flow (baseline)",
        "zorder": 4,
    },
    2022: {
        "color": "red",
        "linewidth": 1.7,
        "alpha": 0.95,
        "label": "2022 daily mean flow (curtailment year)",
        "zorder": 5,
    },
}

for year in comparison_years:
    if year not in daily_year_curves:
        continue

    year_data = daily_year_curves[year]

    # ax_daily.plot(
    #     year_data["plot_date"],
    #     year_data["streamflow_m3s"],
    #     **daily_styles[year],
    # )
    ax_daily.plot(
        year_data["plot_date"],
        year_data["streamflow_cfs"],
        **daily_styles[year],
    )

# ----------------------
# add curtailment dates
# ----------------------
curtailment_date1 = pd.to_datetime("2001-03-15")
curtailment_date2 = pd.to_datetime("2001-08-02")
curtailment_date3 = pd.to_datetime("2001-09-13")

# Add curtailment lines
ax_daily.axvline(
    x=curtailment_date1, 
    color="black",
    linewidth=1,
    linestyle="dotted", 
    label="2022 curtailments"
)
ax_daily.axvline(
    x=curtailment_date2, 
    color="black",
    linewidth=1,
    linestyle="dotted", 
    label="_nolegend_"
)
ax_daily.axvline(
    x=curtailment_date3, 
    color="black",
    linewidth=1,
    linestyle="dotted", 
    label="_nolegend_"
)

    
ax_daily.set_title(
    f"Daily Streamflow Relative to the 2016–2025 Median — USGS {site_no}"
)
# ax_daily.set_ylabel("Daily mean discharge (m³/s)")
ax_daily.set_ylabel("Daily mean discharge (cfs)")
ax_daily.grid(alpha=0.3)
ax_daily.legend(loc="upper right")

# Daily plot: daily data, but month names as tick labels
ax_daily.set_xlim(
    pd.Timestamp("2001-03-01"),
    pd.Timestamp("2001-10-31"),
)
ax_daily.set_ylim(0, 600)

ax_daily.set_xlabel("DOY")

# Tick locations are the first day of each month; labels display month names.
ax_daily.xaxis.set_major_locator(mdates.MonthLocator())
ax_daily.xaxis.set_major_formatter(mdates.DateFormatter("%b"))

# ---- Panel B: Monthly mean streamflow ----

# Faint gray lines: all years other than 2020 and 2022.
for year in plot_years:
    if year in comparison_years or year not in monthly_by_year.columns:
        continue

    if year == 2016:
        ax_monthly.plot(
            monthly_by_year.index,      # Integer months: 3 through 10
            monthly_by_year[year],
            color="lightgray",
            # marker="o",
            # markersize=2,
            linewidth=1.0,
            alpha=0.55,
            zorder=1,
            label="Other years",
        )
    else:
        ax_monthly.plot(
            monthly_by_year.index,      # Integer months: 3 through 10
            monthly_by_year[year],
            color="lightgray",
            # marker="o",
            # markersize=2,
            linewidth=1.0,
            alpha=0.55,
            zorder=1,
            label="_nolegend_",
        )

# Highlight 2020 and 2022.
monthly_styles = {
    2020: {
        "color": "steelblue",
        "linewidth": 2.3,
        # "markersize": 2,
        "label": "2020 (baseline)",
        "zorder": 4,
    },
    2022: {
        "color": "red",
        "linewidth": 2.5,
        # "markersize": 2,
        "label": "2022 (curtailment year)",
        "zorder": 5,
    },
}

for year in comparison_years:
    if year not in monthly_by_year.columns:
        continue

    ax_monthly.plot(
        monthly_by_year.index,
        monthly_by_year[year],
        # marker="o",
        **monthly_styles[year],
    )

# Add curtailment lines
ax_monthly.axvline(
    x=curtailment_date1, 
    color="black",
    linewidth=1,
    linestyle="dotted", 
    label="2022 curtailments"
)
ax_monthly.axvline(
    x=curtailment_date2, 
    color="black",
    linewidth=1,
    linestyle="dotted", 
    label="_nolegend_"
)
ax_monthly.axvline(
    x=curtailment_date3, 
    color="black",
    linewidth=1,
    linestyle="dotted", 
    label="_nolegend_"
)


ax_monthly.set_title(
    f"Monthly Streamflow Volume Comparison (March–October) — USGS {site_no}"
)
ax_monthly.set_xlabel("Month")
ax_monthly.set_ylabel("Streamflow volume (acre-feet/month)")

# Monthly plot: use the same dummy-year date range as the daily plot.
ax_monthly.set_xlim(
    pd.Timestamp("2001-03-01"),
    pd.Timestamp("2001-10-31"),
)

ax_monthly.xaxis.set_major_locator(mdates.MonthLocator())
ax_monthly.xaxis.set_major_formatter(mdates.DateFormatter("%b"))

ax_monthly.grid(alpha=0.3)
ax_monthly.legend(loc="upper right")


# ---- Panel C: Cumulative March–October streamflow volume ----

# Plot all other years as faint gray reference curves.
for year in plot_years:
    if year in comparison_years or year not in cumulative_year_curves:
        continue

    year_data = cumulative_year_curves[year]

    # Give only one gray line a legend entry.
    label = "Other years" if year == min(
        y for y in plot_years if y not in comparison_years
    ) else "_nolegend_"

    ax_cumulative.plot(
        year_data["plot_date"],
        year_data["cumulative_streamflow_acft"],
        color="lightgray",
        linewidth=1.0,
        alpha=0.6,
        zorder=1,
        label=label,
    )

# Highlight 2020 baseline and 2022 curtailment year.
cumulative_styles = {
    2020: {
        "color": "steelblue",
        "linewidth": 2.3,
        "alpha": 1.0,
        "label": "2020 (baseline)",
        "zorder": 4,
    },
    2022: {
        "color": "red",
        "linewidth": 2.5,
        "alpha": 1.0,
        "label": "2022 (curtailment year)",
        "zorder": 5,
    },
}

for year in comparison_years:
    if year not in cumulative_year_curves:
        continue

    year_data = cumulative_year_curves[year]

    ax_cumulative.plot(
        year_data["plot_date"],
        year_data["cumulative_streamflow_acft"],
        **cumulative_styles[year],
    )

# Add curtailment lines
ax_cumulative.axvline(
    x=curtailment_date1, 
    color="black",
    linewidth=1,
    linestyle="dotted", 
    label="2022 curtailments"
)
ax_cumulative.axvline(
    x=curtailment_date2, 
    color="black",
    linewidth=1,
    linestyle="dotted", 
    label="_nolegend_"
)
ax_cumulative.axvline(
    x=curtailment_date3, 
    color="black",
    linewidth=1,
    linestyle="dotted", 
    label="_nolegend_"
)

ax_cumulative.set_title(
    f"Cumulative Streamflow Volume (March–October) — USGS {site_no}"
)
ax_cumulative.set_xlabel("Month")
ax_cumulative.set_ylabel("Cumulative streamflow volume (acre-feet)")

ax_cumulative.set_xlim(plot_start, plot_end)
ax_cumulative.xaxis.set_major_locator(mdates.MonthLocator())
ax_cumulative.xaxis.set_major_formatter(mdates.DateFormatter("%b"))

ax_cumulative.grid(alpha=0.3)
ax_cumulative.legend(loc="upper left")

plt.show()